W poniższym notebooku przedstawiono kod, który przeszukuje foldery i wszystkie podfoldery z plikami *Million Song Dataset* oraz usuwa pliki utworów, które nie znajdują się w *Taste Profile Subset* (porównanie `song_id`).

## Import bibliotek i pliku z piosenkami

In [1]:
import tarfile
import os
import struct
import pandas as pd
import tempfile

In [ ]:
play_count_df = pd.read_csv('train_triplets.txt', sep='\t', header=None, names=['user_id', 'song_id', 'play_count'])

In [6]:
play_count_ids = set(play_count_df['song_id'].astype(str).tolist())

## Nowy kod

In [ ]:
import tarfile
import os

# funckja do rozpakowywania archiwum .tar.gz
def extract_archive(archive_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path=output_dir)
    return output_dir

In [ ]:
import h5py
import pickle
import os

# funkcja do filtrowania plików .h5 na podstawie song_id
def filter_h5_files(folder_path, play_count_ids, processed_file_path=None):
    kept_files = []
    deleted_files = []

    # wczytanie listy już przetworzonych plików
    if processed_file_path and os.path.exists(processed_file_path):
        with open(processed_file_path, "rb") as f:
            processed_files = pickle.load(f)
    else:
        processed_files = set()

    for root, dirs, files in os.walk(folder_path):
        print("Checking folder:", root)
        for filename in files:
            if not filename.endswith(".h5"):
                continue

            file_path = os.path.join(root, filename)

            try:
                with h5py.File(file_path, 'r') as h5:
                    songs_group = h5["metadata"]["songs"]
                    song_id_raw = songs_group[0]['song_id']

                    if isinstance(song_id_raw, bytes):
                        song_id = song_id_raw.decode('utf-8')
                    else:
                        song_id = str(song_id_raw)

                if song_id in play_count_ids:
                    kept_files.append(file_path)
                else:
                    os.remove(file_path)
                    deleted_files.append(file_path)

            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
                continue

            # dodajemy plik do listy przetworzonych
            processed_files.add(file_path)

            # zapis stanu po każdym pliku (opcjonalnie, bezpieczne w przypadku przerwania)
            if processed_file_path:
                with open(processed_file_path, "wb") as f:
                    pickle.dump(processed_files, f)

    return kept_files, deleted_files


In [9]:
folder_to_check = "extracted_files/C"
processed_file = "processed_files_C.pkl"

play_count_ids = set(play_count_df['song_id'].astype(str).tolist())

kept, deleted = filter_h5_files(folder_to_check, play_count_ids, processed_file_path=processed_file)
print(f"Kept: {len(kept)}, Deleted: {len(deleted)}")

Checking folder: extracted_files/C
Checking folder: extracted_files/C\A
Checking folder: extracted_files/C\A\A
Checking folder: extracted_files/C\A\B
Checking folder: extracted_files/C\A\C
Checking folder: extracted_files/C\A\D
Checking folder: extracted_files/C\A\E
Checking folder: extracted_files/C\A\F
Checking folder: extracted_files/C\A\G
Checking folder: extracted_files/C\A\H
Checking folder: extracted_files/C\A\I
Checking folder: extracted_files/C\A\J
Checking folder: extracted_files/C\A\K
Checking folder: extracted_files/C\A\L
Checking folder: extracted_files/C\A\M
Checking folder: extracted_files/C\A\N
Checking folder: extracted_files/C\A\O
Checking folder: extracted_files/C\A\P
Checking folder: extracted_files/C\A\Q
Checking folder: extracted_files/C\A\R
Checking folder: extracted_files/C\A\S
Checking folder: extracted_files/C\A\T
Checking folder: extracted_files/C\A\U
Checking folder: extracted_files/C\A\V
Checking folder: extracted_files/C\A\W
Checking folder: extracted_file

In [10]:
# repeat the code in the loop
directories = ["extracted_files/A", "extracted_files/B", "extracted_files/D", "extracted_files/E"]

for dir_path in directories:
    processed_file = f"processed_files_{os.path.basename(dir_path)}.pkl"
    kept, deleted = filter_h5_files(dir_path, play_count_ids, processed_file_path=processed_file)
    print(f"Directory: {dir_path} - Kept: {len(kept)}, Deleted: {len(deleted)}")

Checking folder: extracted_files/A
Checking folder: extracted_files/A\A
Checking folder: extracted_files/A\A\A
Checking folder: extracted_files/A\A\A\A
Checking folder: extracted_files/A\A\A\B
Checking folder: extracted_files/A\A\A\C
Checking folder: extracted_files/A\A\A\D
Checking folder: extracted_files/A\A\A\E
Checking folder: extracted_files/A\A\A\F
Checking folder: extracted_files/A\A\A\G
Checking folder: extracted_files/A\A\A\H
Checking folder: extracted_files/A\A\A\I
Checking folder: extracted_files/A\A\A\J
Checking folder: extracted_files/A\A\A\K
Checking folder: extracted_files/A\A\A\L
Checking folder: extracted_files/A\A\A\M
Checking folder: extracted_files/A\A\A\N
Checking folder: extracted_files/A\A\A\O
Checking folder: extracted_files/A\A\A\P
Checking folder: extracted_files/A\A\A\Q
Checking folder: extracted_files/A\A\A\R
Checking folder: extracted_files/A\A\A\S
Checking folder: extracted_files/A\A\A\T
Checking folder: extracted_files/A\A\A\U
Checking folder: extracted_f

In [12]:
letters = ['F', 'G', 'H', 'I', 'J', 'K']

for letter in letters:
    archive_path = f"MSD/{letter}.tar.gz"
    output_dir = f"extracted_files/{letter}"
    extract_archive(archive_path, output_dir)
    print(f"Extracted {archive_path} to {output_dir}")

    processed_file = f"processed_files_{letter}.pkl"
    kept, deleted = filter_h5_files(output_dir, play_count_ids, processed_file_path=processed_file)
    print(f"Directory: {output_dir} - Kept: {len(kept)}, Deleted: {len(deleted)}")

Extracted MSD/F.tar.gz to extracted_files/F
Checking folder: extracted_files/F
Checking folder: extracted_files/F\F
Checking folder: extracted_files/F\F\A
Checking folder: extracted_files/F\F\A\A
Checking folder: extracted_files/F\F\A\B
Checking folder: extracted_files/F\F\A\C
Checking folder: extracted_files/F\F\A\D
Checking folder: extracted_files/F\F\A\E
Checking folder: extracted_files/F\F\A\F
Checking folder: extracted_files/F\F\A\G
Checking folder: extracted_files/F\F\A\H
Checking folder: extracted_files/F\F\A\I
Checking folder: extracted_files/F\F\A\J
Checking folder: extracted_files/F\F\A\K
Checking folder: extracted_files/F\F\A\L
Checking folder: extracted_files/F\F\A\M
Checking folder: extracted_files/F\F\A\N
Checking folder: extracted_files/F\F\A\O
Checking folder: extracted_files/F\F\A\P
Checking folder: extracted_files/F\F\A\Q
Checking folder: extracted_files/F\F\A\R
Checking folder: extracted_files/F\F\A\S
Checking folder: extracted_files/F\F\A\T
Checking folder: extracte

KeyboardInterrupt: 

kod przerwany z uwagi na za długie wykonywanie się -> ponowna próba dla folderów J i K

In [13]:
letters = ['J', 'K']

for letter in letters:
    archive_path = f"MSD/{letter}.tar.gz"
    output_dir = f"extracted_files/{letter}"
    extract_archive(archive_path, output_dir)
    print(f"Extracted {archive_path} to {output_dir}")

    processed_file = f"processed_files_{letter}.pkl"
    kept, deleted = filter_h5_files(output_dir, play_count_ids, processed_file_path=processed_file)
    print(f"Directory: {output_dir} - Kept: {len(kept)}, Deleted: {len(deleted)}")

Extracted MSD/J.tar.gz to extracted_files/J
Checking folder: extracted_files/J
Checking folder: extracted_files/J\J
Checking folder: extracted_files/J\J\A
Checking folder: extracted_files/J\J\A\A
Checking folder: extracted_files/J\J\A\B
Checking folder: extracted_files/J\J\A\C
Checking folder: extracted_files/J\J\A\D
Checking folder: extracted_files/J\J\A\E
Checking folder: extracted_files/J\J\A\F
Checking folder: extracted_files/J\J\A\G
Checking folder: extracted_files/J\J\A\H
Checking folder: extracted_files/J\J\A\I
Checking folder: extracted_files/J\J\A\J
Checking folder: extracted_files/J\J\A\K
Checking folder: extracted_files/J\J\A\L
Checking folder: extracted_files/J\J\A\M
Checking folder: extracted_files/J\J\A\N
Checking folder: extracted_files/J\J\A\O
Checking folder: extracted_files/J\J\A\P
Checking folder: extracted_files/J\J\A\Q
Checking folder: extracted_files/J\J\A\R
Checking folder: extracted_files/J\J\A\S
Checking folder: extracted_files/J\J\A\T
Checking folder: extracte

### Logi z kodów

Directory: extracted_files/A - Kept: 14896, Deleted: 24204  
Directory: extracted_files/B - Kept: 14542, Deleted: 23723  
Directory: extracted_files/C - Kept: 14637, Deleted: 4  
Directory: extracted_files/D - Kept: 14901, Deleted: 1  
Directory: extracted_files/E - Kept: 14637, Deleted: 0  
Directory: extracted_files/F - Kept: 14871, Deleted: 24048  
Directory: extracted_files/G - Kept: 14460, Deleted: 23696  
Directory: extracted_files/H - Kept: 14753, Deleted: 23766  
Directory: extracted_files/I - Kept: 14951, Deleted: 23240  
Directory: extracted_files/J - Kept: 14859, Deleted: 23495  
Directory: extracted_files/K - Kept: 14962, Deleted: 23521